In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 15:49:56,126 [DEBUG] [Rain] Rain is initialized
2023-07-07 15:49:56,131 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 15:49:56,136 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\coord/
2023-07-07 15:49:56,142 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 15:49:56,146 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 15:49:56,154 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-07 15:49:56,159 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-07 15:49:56,164 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 15:49:56,626 [INFO] [Provisioner] provisioner is serving
2023-07-07 15:49:56,630 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 15:49:56,637 [INFO] [Coordinator] coordinator is serving
2023-07-07 15:49:56,641 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 15:49:56,669 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 15:49:56,678 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 15:49:56,682 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 15:49:56,686 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\worker_50151/
2023-07-07 15:49:56,701 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 15:49:56,705 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\worker_50152/
2023-07-07 15:49:56,716 [INFO] [Worker_50152] Worker is running on port: 50152
2023-0

Epoch 1/5


2023-07-07 15:50:05,238 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-07 15:50:05,278 [DEBUG] [DividerAmbassador] divider begins executing iteration1 for worker3
2023-07-07 15:50:05,285 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1


Epoch 1/5
Epoch 1/5
157/157 [==============================] - 12s 21ms/step - loss: 0.6850 - accuracy: 0.7840
Epoch 2/5
Epoch 2/5
157/157 [==============================] - 3s 21ms/step - loss: 0.3072 - accuracy: 0.9072
Epoch 3/5
157/157 [==============================] - 3s 21ms/step - loss: 0.2993 - accuracy: 0.9126
Epoch 3/5
157/157 [==============================] - 3s 22ms/step - loss: 0.3068 - accuracy: 0.9075
Epoch 3/5
157/157 [==============================] - 3s 21ms/step - loss: 0.2345 - accuracy: 0.9299
Epoch 4/5
157/157 [==============================] - 3s 21ms/step - loss: 0.2287 - accuracy: 0.9311
Epoch 4/5
157/157 [==============================] - 3s 21ms/step - loss: 0.1962 - accuracy: 0.9405
Epoch 5/5
157/157 [==============================] - 3s 20ms/step - loss: 0.1907 - accuracy: 0.9427
Epoch 5/5
157/157 [==============================] - 3s 20ms/step - loss: 0.1965 - accuracy: 0.9398
Epoch 5/5
121/157 [======================>.......] - ETA: 0s - loss: 0.1708 - a

2023-07-07 15:50:32,122 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2


...

2023-07-07 15:53:20,582 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 15:53:20,590 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/1_3_trained.pkl from worker1


sending data to divider
 62/157 [==========>...................] - ETA: 0s - loss: 0.0830 - accuracy: 0.9729

2023-07-07 15:53:20,796 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData\divider/1_3_trained.pkl from worker1 successfully


157/157 [==============================] - 1s 8ms/step - loss: 0.0826 - accuracy: 0.9740
sending data to dividersending data to divider



2023-07-07 15:53:21,496 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-07 15:53:21,496 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 15:53:21,499 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 15:53:21,500 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData\divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData\divider/2_3_trained.pkl from worker2
2023-07-07 15:53:21,595 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData\divider/2_3_trained.pk

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 3ms/step - loss: 0.0708 - accuracy: 0.9786

Test accuracy: 97.9%
